# Тестовое задание

**Постановка задачи:**<br>
необходимо реализовать рекомендательную систему на кастомном датасете медиа контента.<br> *Будем рады если покажите нам все свои лучшие стороны через решение этого задания* <br><br>
Задание составлено с целью понять что у вас есть базовое представление о том чем вам придётся заниматься и желание расти в этом направлении так что если хотите блеснуть сложными алгоритмами например <a link='https://www.ijcai.org/Proceedings/2017/0447.pdf'>Deep Matrix Factorization</a> то подождите до технического собеседования


## Дополнительная информация 

**Из каких критериев складывается наше мнение о работе?**
1. Адекватность работы модели как по качеству, так и по скорости работы в проде
2. Качество написания кода: стиль кода, конструкции, эффективность работы
3. Обоснованность и логичность принятых решений
4. Скорость вашей работы. Чем быстрее вы пришлёте задание, тем быстрее мы пригласим вас на следующий раунд
**В какие сроки мы ожидаем получить от вас работу?**<br>
На задание даётся 3 дня с момента старта, отсчёт времени начитает с времени отправки задания<br><br>
**Какие шаги мы бы хотели увидеть в решение?**<br>
для успешного прохождения задания мы бы хотели понять не только Ваш код и ход Ваших рассуждений для этому мы хотели бы увидеть от вас несколько этапов<br>
1. Сформировать постановку задачу (какую задачу решаете?, какую ошибку минимизируете?) <br>
3. Сформировать baseline (эвристику) какой вам кажется логичным для этой задачи<br>
4. Обучить модель и объяснить свой выбор<br>
5. Провести валидацию модели, оценить стандартные метрики качества<br> 
<br> 
5. Определить бизнес метрику, по которой вы понимаете модель хорошая или плохая (необязательный пункт, будет плюсом)<br>
5. Оцените эффект от вашей модели, почему вы считаете, что ваша модель хорошая или плохая? (необязательный пункт, будет плюсом)<br>

**Какие методы я могу применять и что лучше использовать?**<br>
Какие вам хочется если они обоснованы<br><br>

**Куда нужно отправить работу?**<br>
Необходимо залить ноутбук в гит и скинуть на него ссылку на почту mshmonov@sbdagroup.com<br>
Заголовок письма: "тестовое задание: Иванов Иван Дата"<br>
в теле письме файл с аналогичным названием "тестовое задание: Иванов Иван Дата" <br>

**Что будет дальше?**<br>
В срок до 3 дней мы вернёмся с обратной связью по первому этапу. После прохождения будет ещё два собеседования<br>
1. Техническое собеседование, обсуждаем используемые технологии начиная от знаний питона заканчивая особенностями ML  алгоритмов и релевантностью опыта кандидата, если вы получили это задание то нам уже понравился Ваш опыт и мы хотим его подтвердить:)<br>
2. Бизнесовое собеседование, решаем бизнес-кейс который сводится к формализованной задачи машинного обучения и обсуждаем возможные пути её решения<br>

## Загрузка данных

In [62]:
import pandas as pd
import numpy as np

In [63]:
df_films = pd.read_csv('./data/films.csv')
df_ratings = pd.read_csv('./data/ratings.csv')

C:\Users\frida\anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3418: DtypeWarning: Columns (4) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


Для решения данной задачи ришил вручную нагенерировать пользователей(каждый допустим смотрел 50 фильмов). Так же смотрел 
схожести фильмов(по описанию самих фильмов). Хотел бы сделать так со всеми признаками фильмов, но это долго будет считаться.
Логика была бы такой же. Взял бы tf-idf меру, посмортрел бы косинусную схожесть. Далее все матрицы складываются со своими весами.
#Делал такое решение, так как у нас были данные о рейтингах, которые проставили пользователи. На их основе я генерировал пользователей. 
И так же были описания самих фильмов. В качестве входного параметра можно брать просто самые похожие фильмы(если пользователь
не зарегестрирован, например). А так же можно отправлять вектор(просмотренные фильмы и рейтинги, которые оставил пользователь).
Так же будет выдаваться список лучших фильмов.
Так как данные о пользователях сгенерированы рандомно, тестировку модели не проводил. Для реальных пользователей можно было
бы оценить модель сравнив выдачу модели с рейтингом, проставленным пользователем. В моей модели рейтинг выставлен наугад, связи 
нет, поэтому тяжело отслеживать скор. 

In [64]:
df_ratings.fillna(0,inplace = True)

In [67]:
ages = [0,18,30,45]
column_names = []
for i in ages:
    column_names.append('males_'+ str(i) + 'age_avg_vote')
    column_names.append('females_'+ str(i) + 'age_avg_vote')
column_names_for = []
for i in ages:
    df_ratings['males_'+str(i)+'propability'] = df_ratings['males_'+ str(i) + 'age_votes'] / df_ratings['total_votes'] 
    df_ratings['females_'+str(i)+'propability'] = df_ratings['females_'+ str(i) + 'age_votes'] / df_ratings['total_votes']
    column_names_for.append('males_'+str(i)+'propability')
    column_names_for.append('females_'+str(i)+'propability')

In [68]:
import random
from scipy.sparse import lil_matrix

In [73]:
matrix = lil_matrix((len(range(4999)), len(df_films)))
for i in tqdm(range(4999)):
    res = random.sample(range(0, len(df_films)), 50)
    for j in res:
        rand = random.randint(0,7)
        name = df_films.iloc[j]['title']
        title_id = df_films.iloc[j]['title_id']
        result =  df_ratings.loc[df_ratings['title_id'] == title_id][column_names[rand]]
        matrix[i,j] = list(result)[0]

100%|██████████████████████████████████████████████████████████████████████████████| 4999/4999 [33:17<00:00,  2.50it/s]


In [111]:
matrix = matrix.T
percent = float(matrix.nnz) / 5000 / 78343 * 100
print(u"Процент заполненности матрицы: %.2f%%" % percent)

Процент заполненности матрицы: 0.05%


In [112]:
from sklearn.preprocessing import normalize
from scipy.sparse import spdiags

# косинусная мера вычисляется как отношение скалярного произведения векторов(числитель) 
# к произведению длины векторов(знаменатель)

# нормализуем исходную матрицу 
# (данное действие соответствует приведению знаменателя в формуле косинусной меры к 1)
normalized_matrix = normalize(matrix.tocsr()).tocsr()
# вычисляем скалярное произведение
cosine_sim_matrix_ratings = normalized_matrix.dot(normalized_matrix.T)

# обнуляем диагональ, чтобы исключить ее из рекомендаций
# быстрое обнуление диагонали
diag = spdiags(-cosine_sim_matrix_ratings.diagonal(), [0], *cosine_sim_matrix_ratings.shape, format='csr')
cosine_sim_matrix_ratings = cosine_sim_matrix_ratings + diag

percent = float(cosine_sim_matrix_ratings.nnz) / cosine_sim_matrix_ratings.shape[0] / cosine_sim_matrix_ratings.shape[1] * 100
print(u"Процент заполненности матрицы: %.2f%%" % percent)
print("Размер в МБ:", cosine_sim_matrix_ratings.data.nbytes / 1024 / 1024)

Процент заполненности матрицы: 0.13%
Размер в МБ: 61.23431396484375


### description i2i matrix

In [76]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import lil_matrix, csr_matrix, spdiags,vstack,save_npz,load_npz
from tqdm.notebook import tqdm

In [77]:
df_films = df_films.reset_index(drop=True)

In [78]:
tf_idf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tf_idf.fit_transform(df_films['description'])
percent = float(tfidf_matrix.nnz) / tfidf_matrix.shape[0] / tfidf_matrix.shape[1] * 100
print (u"Процент заполненности матрицы: %.2f%%" % percent)
print (u"Размер в МБ:", tfidf_matrix.data.nbytes / 1024 / 1024)
tfidf_matrix

Процент заполненности матрицы: 0.02%
Размер в МБ: 8.598564147949219


<78343x66163 sparse matrix of type '<class 'numpy.float64'>'
	with 1127031 stored elements in Compressed Sparse Row format>

In [79]:
%%time
top_k = 100 #оставляем k наиболее похожих фильмов, для остальных схожесть = 0
batch_size = 5000 #считаем схожесть поэтапно, для снижения затрат памяти

import time
start_time = time.time()

cosine_sim_matrix = None
for i in tqdm(range(0, tfidf_matrix.shape[0], batch_size)):
    right_board = min(tfidf_matrix.shape[0], i+batch_size)
    cosine_sim_rows = csr_matrix(cosine_similarity(tfidf_matrix[i:right_board],tfidf_matrix))
    for j in range(cosine_sim_rows.shape[0]):
        row = np.zeros(tfidf_matrix.shape[0])
        cosine_sim_row = cosine_sim_rows[j]
        row[np.argsort(cosine_sim_row.toarray()[0])[-top_k+1:-1][::-1]] = np.sort(cosine_sim_row.toarray()[0])[-top_k+1:-1][::-1]
        sparse_top_k_films = csr_matrix(row, dtype = np.float32)
        if cosine_sim_matrix is None:
            cosine_sim_matrix = sparse_top_k_films
        else:
            cosine_sim_matrix = vstack((cosine_sim_matrix, sparse_top_k_films))
        
    if (i!=0)and((i+batch_size)%10000==0):
        percent = float(cosine_sim_matrix.nnz) / cosine_sim_matrix.shape[0] / cosine_sim_matrix.shape[1] * 100
        print (f"Обработано элементов: {i+batch_size} из {tfidf_matrix.shape[0]}" )
        print (u"Процент заполненности матрицы: %.2f%%" % percent)
        print (u"Размер в МБ:", cosine_sim_matrix.data.nbytes / 1024 / 1024)
        print(f"--- %{(time.time() - start_time)/60} minutes ---")
        print (f"-----------------------------------------------------" )

  0%|          | 0/16 [00:00<?, ?it/s]

Обработано элементов: 10000 из 78343
Процент заполненности матрицы: 0.12%
Размер в МБ: 3.7337417602539062
--- %1.673899511496226 minutes ---
-----------------------------------------------------
Обработано элементов: 20000 из 78343
Процент заполненности матрицы: 0.12%
Размер в МБ: 7.466239929199219
--- %3.5091222961743673 minutes ---
-----------------------------------------------------
Обработано элементов: 30000 из 78343
Процент заполненности матрицы: 0.12%
Размер в МБ: 11.20071029663086
--- %5.8512097279230755 minutes ---
-----------------------------------------------------
Обработано элементов: 40000 из 78343
Процент заполненности матрицы: 0.12%
Размер в МБ: 14.935585021972656
--- %8.603630872567495 minutes ---
-----------------------------------------------------
Обработано элементов: 50000 из 78343
Процент заполненности матрицы: 0.12%
Размер в МБ: 18.669200897216797
--- %11.742301245530447 minutes ---
-----------------------------------------------------
Обработано элементов: 60

In [149]:
#тестим схожесть
film_index = 70000
data_ids = np.argsort(cosine_sim_matrix_ratings[film_index].toarray())

display(df_films[df_films.index==film_index])
display(df_films[df_films.index.isin(data_ids)])

print(np.argsort(cosine_sim_matrix_ratings[film_index].toarray()[0])[-1])
print(cosine_similarity(tfidf_matrix[film_index],tfidf_matrix[np.argsort(cosine_sim_matrix_ratings[film_index].toarray()[0])[-1]])) 

,title_id,title,original_title,year,date_published,genre,duration,country,language,director,writer,production_company,actors,description,avg_vote,votes,уear
70000,tt4993990,Die Ontwaking,Die Ontwaking,2015,2016-02-26,"Crime, Horror, Thriller",97,South Africa,Afrikaans,Johnny Breedt,"Johnny Breedt, Chris Karsten",Advantage Entertainment,"Gys de Villiers, Juanita de Villiers, Gérard R...","A gruesome and action-packed crime thriller. ""...",4.4,111,2015


,title_id,title,original_title,year,date_published,genre,duration,country,language,director,writer,production_company,actors,description,avg_vote,votes,уear
64015,tt3079568,Minutes to Midnight,Minutes to Midnight,2018,2018-07-03,"Action, Horror",91,USA,English,Christopher Ray,"Victoria Dadi, Christopher M. Don",DeInstitutionalized,"William Baldwin, Richard Grieco, Bill Moseley,...","On the cusp of New Year's Eve, seven friends a...",3.4,268,2018


64015
[[0.]]


In [154]:
#тестим схожесть
film_index = 70000
data_ids = np.argsort(cosine_sim_matrix_ratings[film_index].toarray()[0])
data_ids = data_ids[-20:]
display(df_films[df_films.index==film_index])
df_films[df_films.index.isin(data_ids)]

,title_id,title,original_title,year,date_published,genre,duration,country,language,director,writer,production_company,actors,description,avg_vote,votes,уear
70000,tt4993990,Die Ontwaking,Die Ontwaking,2015,2016-02-26,"Crime, Horror, Thriller",97,South Africa,Afrikaans,Johnny Breedt,"Johnny Breedt, Chris Karsten",Advantage Entertainment,"Gys de Villiers, Juanita de Villiers, Gérard R...","A gruesome and action-packed crime thriller. ""...",4.4,111,2015


,title_id,title,original_title,year,date_published,genre,duration,country,language,director,writer,production_company,actors,description,avg_vote,votes,уear
1021,tt0021750,Le vie della città,City Streets,1931,1932-03-07,"Crime, Drama, Film-Noir",83,USA,English,Rouben Mamoulian,"Dashiell Hammett, Max Marcin",Paramount Pictures,"Gary Cooper, Sylvia Sidney, Paul Lukas, Willia...",Man joins a gang to free his girlfriend from p...,7.1,1330,1931
15067,tt0068288,Blood Orgy of the She-Devils,Blood Orgy of the She-Devils,1973,1973-01-01,"Horror, Thriller",73,USA,English,Ted V. Mikels,Ted V. Mikels,Occult Films,"Lila Zaborin, Victor Izay, Tom Pace, Leslie Mc...",Lorraine and Mark enter the world of witchcraf...,3.2,614,1973
16166,tt0071487,Il fantasma della libertà,Le fantôme de la liberté,1974,1974-11-23,Comedy,104,"France, Italy",French,Luis Buñuel,"Luis Buñuel, Jean-Claude Carrière",Greenwich Film Productions,"Adriana Asti, Julien Bertheau, Jean-Claude Bri...",A series of surreal sequences that critique mo...,7.9,13903,1974
31209,tt0139861,Arunachalam,Arunachalam,1999,1997-04-10,"Action, Comedy, Drama",165,India,Tamil,Sundar C.,Crazy Mohan,Annamalai Cine Combines,"Rajinikanth, Soundarya, Rambha, Jaishankar, Ra...",The movie revolves around Arunachalam (Rajini)...,7.4,1860,1999
34303,tt0203427,Le créateur,Le créateur,1999,1999-06-16,Comedy,90,France,French,Albert Dupontel,"Albert Dupontel, Gilles Laurent",Canal+,"Claude Perron, Albert Dupontel, Philippe Uchan...","A successful author, Darius sees posters annou...",6.9,945,1999
34584,tt0210070,Licantropia Evolution - Ritorno al presente,Ginger Snaps,2000,2001-05-11,"Drama, Fantasy, Horror",108,Canada,English,John Fawcett,"Karen Walton, John Fawcett",Copperheart Entertainment,"Emily Perkins, Katharine Isabelle, Kris Lemche...","Two death-obsessed sisters, outcasts in their ...",6.8,40897,2000
37246,tt0277944,Panic,Panic,2002,2002-01-22,"Action, Drama, Thriller",91,USA,English,Bob Misiorowski,"Jace Anderson, Boaz Davidson",Millennium Films,"Rodney Rowland, Kristanna Loken, Alexander Enb...",FAA system analyzer named Neil McCabe is the o...,3.6,715,2002
37694,tt0286493,The Badge,The Badge,2002,2002-09-07,"Crime, Drama, Mystery",103,USA,English,Robby Henson,Robby Henson,Emmett/Furla/Oasis Films (EFO Films),"Billy Bob Thornton, Patricia Arquette, William...",The sheriff investigates a tipped over truck f...,6.1,3140,2002
40154,tt0355857,Onmyôji,Onmyôji,2001,2001-10-06,"Action, Drama, Fantasy",112,Japan,Japanese,Yôjirô Takita,Baku Yumemakura,Dentsu,"Mansai Nomura, Hideaki Itô, Eriko Imai, Yui Na...","When samurai Hiromasa comes to Seimei, Kyoto's...",6.3,1269,2001
45439,tt0492754,Gruesome,Salvage,2006,2006-01-19,"Horror, Mystery, Thriller",80,USA,English,"Jeff Crook, Josh Crook","Josh Crook, Jeff Crook",Crook Brothers Productions,"Lauren Currie Lewis, Chris Ferry, Cody Darbe, ...",Claire Parker is going to die. At the hands of...,5.4,2493,2006


### конец секции